# 03 — Deep Data Mining: Homophily, Community Structure & Anomaly Detection

We investigate whether neural network attribution graphs exhibit structural
patterns associated with interpretability using advanced graph mining.

**Key analyses:**
1. Feature-type homophily — do nodes cluster by type?
2. Community detection and comparison across graph types
3. Motif analysis — recurring structural patterns
4. Graph anomaly detection
5. Layer-wise flow analysis


In [1]:
import sys
sys.path.insert(0, "..")

import importlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from scipy import stats
from pathlib import Path
from collections import Counter, defaultdict

import src.graph_generator
import src.structural_metrics
import src.utils
importlib.reload(src.graph_generator)
importlib.reload(src.structural_metrics)
importlib.reload(src.utils)

from src.graph_generator import AttributionGraph
from src.structural_metrics import (
    compute_all_metrics, compute_metrics_batch, attribution_to_networkx, StructuralMetrics
)
from src.dataset import load_graphs_from_dir

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120
print("Imports loaded.")


Imports loaded.


## 1. Load All Graphs


In [2]:
synthetic_graphs = load_graphs_from_dir("../data/raw/synthetic")
real_graphs = []
categories = ["factual_recall", "reasoning", "creative_writing",
              "code_understanding", "ambiguous_context", "multilingual"]
for cat in categories:
    cat_dir = Path(f"../data/raw/{cat}")
    if cat_dir.exists():
        cat_graphs = load_graphs_from_dir(str(cat_dir))
        for g in cat_graphs:
            g.metadata["category"] = cat
        real_graphs.extend(cat_graphs)

all_graphs = synthetic_graphs + real_graphs
print(f"Loaded {len(synthetic_graphs)} synthetic + {len(real_graphs)} real = {len(all_graphs)} total")


Loaded 375 graphs from ../data/raw/synthetic
Loaded 5 graphs from ../data/raw/factual_recall
Loaded 5 graphs from ../data/raw/reasoning
Loaded 0 graphs from ../data/raw/creative_writing
Loaded 0 graphs from ../data/raw/code_understanding
Loaded 0 graphs from ../data/raw/ambiguous_context
Loaded 0 graphs from ../data/raw/multilingual
Loaded 375 synthetic + 10 real = 385 total


## 2. Feature-Type Homophily Analysis

Homophily measures the tendency of nodes to connect with similar nodes.
In attribution graphs, we measure whether feature nodes at similar layers
tend to connect more strongly. We hypothesize that interpretable graphs show
higher layer homophily (clean layer-by-layer flow), while superposed graphs
show lower homophily (features connect across distant layers).


In [3]:
def compute_layer_homophily(graph):
    """Compute layer-based homophily: fraction of edges between same or adjacent layers."""
    G = attribution_to_networkx(graph)
    if G.number_of_edges() == 0:
        return {"layer_homophily": 0.0, "exact_layer_homophily": 0.0, "avg_layer_distance": 0.0}
    
    layers = nx.get_node_attributes(G, "layer")
    same_or_adjacent = 0
    exact_same = 0
    total = 0
    distances = []
    
    for u, v in G.edges():
        if u in layers and v in layers:
            dist = abs(layers[u] - layers[v])
            distances.append(dist)
            if dist <= 1:
                same_or_adjacent += 1
            if dist == 0:
                exact_same += 1
            total += 1
    
    return {
        "layer_homophily": same_or_adjacent / total if total > 0 else 0.0,
        "exact_layer_homophily": exact_same / total if total > 0 else 0.0,
        "avg_layer_distance": np.mean(distances) if distances else 0.0,
    }

def compute_type_homophily(graph):
    """Compute node-type homophily: fraction of edges between same node types."""
    G = attribution_to_networkx(graph)
    if G.number_of_edges() == 0:
        return {"type_homophily": 0.0}
    
    types = nx.get_node_attributes(G, "node_type")
    same_type = sum(1 for u, v in G.edges() if types.get(u) == types.get(v))
    return {"type_homophily": same_type / G.number_of_edges()}


homophily_data = []
for g in all_graphs:
    row = {"prompt": g.prompt[:50], "label": g.metadata.get("label", None),
           "type": g.metadata.get("type", "real")}
    row.update(compute_layer_homophily(g))
    row.update(compute_type_homophily(g))
    homophily_data.append(row)

df_homophily = pd.DataFrame(homophily_data)
print(f"Computed homophily for {len(df_homophily)} graphs")
df_homophily.groupby("type")[["layer_homophily", "exact_layer_homophily", "avg_layer_distance", "type_homophily"]].mean()


Computed homophily for 385 graphs


,layer_homophily,exact_layer_homophily,avg_layer_distance,type_homophily
type,,,,
clean_tree,1.000000,0.000000,1.000000,0.299438
mixed,0.479910,0.119958,2.192593,1.000000
real,0.064506,0.007423,151.534539,0.429925
tangled,0.441588,0.161866,1.954704,1.000000


In [4]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, metric in enumerate(["layer_homophily", "avg_layer_distance", "type_homophily"]):
    ax = axes[i]
    for gtype, color, label in [("clean_tree", "#2ecc71", "Clean"),
                                 ("tangled", "#e74c3c", "Tangled"),
                                 ("mixed", "#f39c12", "Mixed")]:
        subset = df_homophily[df_homophily["type"] == gtype]
        if len(subset) > 0:
            ax.hist(subset[metric], bins=20, alpha=0.6, color=color, label=label, density=True)
    
    real_subset = df_homophily[df_homophily["type"] == "real"]
    if len(real_subset) > 0:
        ax.hist(real_subset[metric], bins=10, alpha=0.7, color="#3498db", label="Real", density=True)
    
    ax.set_title(metric.replace("_", " ").title())
    ax.legend(fontsize=8)

plt.suptitle("Homophily Analysis: Interpretable vs Uninterpretable Graphs", fontsize=13)
plt.tight_layout()
plt.savefig("../results/figures/homophily_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved homophily_analysis.png")


Saved homophily_analysis.png


/var/folders/cp/byf3wc5s7wjd22v0snjtpblh0000gn/T/ipykernel_25324/251776461.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Community Detection & Structure

Detect communities in each graph and analyze how community structure
differs between interpretable and uninterpretable circuits.


In [5]:
from networkx.algorithms.community import greedy_modularity_communities, louvain_communities

def analyze_communities(graph):
    """Detailed community analysis for a single graph."""
    G = attribution_to_networkx(graph)
    G_und = G.to_undirected()
    
    if G_und.number_of_nodes() < 3:
        return {"n_communities": 1, "modularity": 0.0, "community_size_std": 0.0,
                "inter_community_edges": 0.0, "largest_community_frac": 1.0}
    
    try:
        communities = list(louvain_communities(G_und, seed=42))
        mod = nx.algorithms.community.modularity(G_und, communities)
    except Exception:
        communities = list(greedy_modularity_communities(G_und))
        mod = nx.algorithms.community.modularity(G_und, communities)
    
    sizes = [len(c) for c in communities]
    n_nodes = G_und.number_of_nodes()
    
    # Count inter-community edges
    node_to_comm = {}
    for i, comm in enumerate(communities):
        for node in comm:
            node_to_comm[node] = i
    
    inter_edges = sum(1 for u, v in G_und.edges() 
                      if node_to_comm.get(u, -1) != node_to_comm.get(v, -2))
    total_edges = G_und.number_of_edges()
    
    return {
        "n_communities": len(communities),
        "modularity": mod,
        "community_size_std": np.std(sizes) / np.mean(sizes) if np.mean(sizes) > 0 else 0,
        "inter_community_edge_ratio": inter_edges / total_edges if total_edges > 0 else 0,
        "largest_community_frac": max(sizes) / n_nodes if n_nodes > 0 else 0,
    }

community_data = []
for g in all_graphs:
    row = {"label": g.metadata.get("label", None), "type": g.metadata.get("type", "real")}
    row.update(analyze_communities(g))
    community_data.append(row)

df_comm = pd.DataFrame(community_data)
print("Community analysis complete.")
df_comm.groupby("type")[["n_communities", "modularity", "inter_community_edge_ratio", "largest_community_frac"]].mean()


Community analysis complete.


,n_communities,modularity,inter_community_edge_ratio,largest_community_frac
type,,,,
clean_tree,12.020000,0.823434,0.094598,0.142627
mixed,5.453333,0.637445,0.314765,0.335329
real,6.900000,0.573390,0.779737,0.289443
tangled,5.460000,0.221881,0.655927,0.260267


In [6]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, metric in enumerate(["modularity", "n_communities", "inter_community_edge_ratio"]):
    ax = axes[i]
    for gtype, color, label in [("clean_tree", "#2ecc71", "Clean"),
                                 ("tangled", "#e74c3c", "Tangled"),
                                 ("mixed", "#f39c12", "Mixed")]:
        subset = df_comm[df_comm["type"] == gtype]
        if len(subset) > 0:
            ax.hist(subset[metric], bins=20, alpha=0.6, color=color, label=label, density=True)
    ax.set_title(metric.replace("_", " ").title())
    ax.legend(fontsize=8)

plt.suptitle("Community Structure: Interpretable vs Uninterpretable", fontsize=13)
plt.tight_layout()
plt.savefig("../results/figures/community_analysis.png", dpi=150, bbox_inches="tight")
plt.show()


/var/folders/cp/byf3wc5s7wjd22v0snjtpblh0000gn/T/ipykernel_25324/2459553610.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Motif Analysis

Count common subgraph patterns (motifs) in each graph type.
Motifs reveal recurring structural building blocks.


In [7]:
def count_triadic_motifs(graph):
    """Count directed triadic motifs (census)."""
    G = attribution_to_networkx(graph)
    if G.number_of_nodes() < 3:
        return {}
    try:
        census = nx.triadic_census(G)
        total = sum(census.values())
        if total > 0:
            return {k: v / total for k, v in census.items()}
        return census
    except Exception:
        return {}


motif_data = {"clean_tree": [], "tangled": [], "mixed": []}
for g in synthetic_graphs[:60]:
    gtype = g.metadata.get("type", "unknown")
    if gtype in motif_data:
        motifs = count_triadic_motifs(g)
        if motifs:
            motif_data[gtype].append(motifs)

motif_summary = {}
for gtype, motif_list in motif_data.items():
    if motif_list:
        df_motifs = pd.DataFrame(motif_list)
        motif_summary[gtype] = df_motifs.mean()

if motif_summary:
    df_motif_summary = pd.DataFrame(motif_summary)
    if "clean_tree" in df_motif_summary.columns and "tangled" in df_motif_summary.columns:
        df_motif_summary["diff"] = abs(df_motif_summary["clean_tree"] - df_motif_summary["tangled"])
        top_motifs = df_motif_summary.sort_values("diff", ascending=False).head(8)
        print("Top discriminating triadic motifs:")
        print(top_motifs[["clean_tree", "tangled", "mixed", "diff"]])


In [8]:
if motif_summary and "clean_tree" in motif_summary and "tangled" in motif_summary:
    df_plot = df_motif_summary.sort_values("diff", ascending=False).head(8).drop(columns=["diff"])
    
    fig, ax = plt.subplots(figsize=(10, 6))
    df_plot.plot(kind="bar", ax=ax, alpha=0.8,
                 color=["#2ecc71", "#e74c3c", "#f39c12"])
    ax.set_title("Top Discriminating Triadic Motifs", fontsize=13)
    ax.set_ylabel("Normalized Frequency")
    ax.set_xlabel("Motif Type")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig("../results/figures/motif_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Motif data not available for comparison.")


Motif data not available for comparison.


## 5. Layer-wise Information Flow Analysis

Analyze how attribution weight flows through layers.
Interpretable circuits should show clean forward flow;
superposed circuits should show chaotic multi-directional flow.


In [9]:
def compute_flow_matrix(graph, n_layer_bins=10):
    """Compute a layer-to-layer flow matrix (weighted edge sum between layer bins)."""
    G = attribution_to_networkx(graph)
    layers = nx.get_node_attributes(G, "layer")
    if not layers:
        return np.zeros((n_layer_bins, n_layer_bins))
    
    all_layers = list(layers.values())
    normal_layers = [l for l in all_layers if 0 <= l < 100]
    if not normal_layers:
        return np.zeros((n_layer_bins, n_layer_bins))
    
    max_layer = max(normal_layers) + 1
    bin_size = max(1, max_layer / n_layer_bins)
    
    flow = np.zeros((n_layer_bins, n_layer_bins))
    for u, v, data in G.edges(data=True):
        l_u = layers.get(u, 0)
        l_v = layers.get(v, 0)
        if 0 <= l_u < 100 and 0 <= l_v < 100:
            bin_u = min(int(l_u / bin_size), n_layer_bins - 1)
            bin_v = min(int(l_v / bin_size), n_layer_bins - 1)
            flow[bin_u, bin_v] += abs(data.get("weight", 1.0))
    total = flow.sum()
    if total > 0:
        flow /= total
    return flow



flow_by_type = defaultdict(list)
for g in synthetic_graphs:
    gtype = g.metadata.get("type", "unknown")
    flow = compute_flow_matrix(g)
    flow_by_type[gtype].append(flow)

for g in real_graphs:
    flow = compute_flow_matrix(g)
    flow_by_type["real"].append(flow)

avg_flows = {k: np.mean(v, axis=0) for k, v in flow_by_type.items() if v}
print("Flow matrices computed for:", list(avg_flows.keys()))


Flow matrices computed for: ['clean_tree', 'tangled', 'mixed', 'real']


In [10]:
n_types = len(avg_flows)
fig, axes = plt.subplots(1, min(n_types, 4), figsize=(5 * min(n_types, 4), 4))
if n_types == 1:
    axes = [axes]

for ax, (gtype, flow) in zip(axes, avg_flows.items()):
    im = ax.imshow(flow, cmap="YlOrRd", aspect="auto")
    ax.set_title(f"{gtype.replace(chr(95), chr(32)).title()}")
    ax.set_xlabel("Target Layer Bin")
    ax.set_ylabel("Source Layer Bin")
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle("Layer-to-Layer Attribution Flow Patterns", fontsize=13)
plt.tight_layout()
plt.savefig("../results/figures/flow_analysis.png", dpi=150, bbox_inches="tight")
plt.show()


/var/folders/cp/byf3wc5s7wjd22v0snjtpblh0000gn/T/ipykernel_25324/2390265966.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Combine All Advanced Metrics & Save


In [11]:
df_advanced = df_homophily.copy()
for col in ["n_communities", "modularity", "inter_community_edge_ratio", "largest_community_frac"]:
    df_advanced[col] = df_comm[col]

df_advanced.to_csv("../results/metrics/advanced_mining_metrics.csv", index=False)
print(f"Saved {len(df_advanced)} rows to results/metrics/advanced_mining_metrics.csv")
print("\nStatistical Significance (Clean vs Tangled):")
print("=" * 65)
clean = df_advanced[df_advanced["type"] == "clean_tree"]
tangled = df_advanced[df_advanced["type"] == "tangled"]

for col in ["layer_homophily", "avg_layer_distance", "type_homophily", 
            "modularity", "inter_community_edge_ratio"]:
    t, p = stats.ttest_ind(clean[col].dropna(), tangled[col].dropna(), equal_var=False)
    sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
    print(f"  {col:40s} t={t:8.3f}  p={p:.2e}  {sig}")


Saved 385 rows to results/metrics/advanced_mining_metrics.csv

Statistical Significance (Clean vs Tangled):
  layer_homophily                          t= 197.610  p=3.30e-182  ***
  avg_layer_distance                       t= -86.269  p=4.34e-129  ***
  type_homophily                           t=-750.987  p=1.73e-268  ***
  modularity                               t= 209.526  p=2.41e-245  ***
  inter_community_edge_ratio               t=-164.504  p=5.51e-265  ***


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
